# Models

And now - this colab unveils the heart (or the brains?) of the transformers library - the models:

https://colab.research.google.com/drive/1hhR9Z-yiqjUe7pJjVQw4c74z_V3VchLy?usp=sharing

This should run nicely on a low-cost or free T4 box.

In [ ]:
quant_config = BitsAndBytesConfig(...)

You’re instantiating a configuration object that tells the runtime how to load and run a model in low precision (4-bit) instead of full precision (e.g., FP16/FP32).

This is mainly about:

Reducing VRAM usage
Enabling large models to fit on smaller GPUs
Accepting a small tradeoff in numerical precision
load_in_4bit=True

This is the switch that turns on 4-bit quantization.

Model weights are stored in 4 bits per parameter instead of:
16-bit (FP16 / BF16)
32-bit (FP32)
Memory savings: ~75% compared to FP16

👉 Without this, the rest of the settings are ignored.

bnb_4bit_use_double_quant=True

This enables double quantization (a second layer of compression).

First: weights → 4-bit
Second: quantization constants themselves → further compressed

Why this matters:

Reduces memory even more
Slight extra compute overhead
Usually negligible impact on accuracy

Think of it as:

“compress the compression”

bnb_4bit_compute_dtype=torch.bfloat16

This controls the compute precision, not storage.

Even though weights are stored in 4-bit, operations happen in bfloat16
bfloat16:
Similar range to FP32
Less precise mantissa than FP32, but more stable than FP16 in many cases

Why this is important:

Prevents numerical instability during inference/training
Good balance between:
speed
stability
hardware compatibility (especially on newer GPUs/TPUs)
bnb_4bit_quant_type="nf4"

This selects the quantization scheme.

"nf4" = Normal Float 4
Specifically designed for neural network weights

Compared to standard 4-bit:

Better represents values with a normal (Gaussian-like) distribution
Improves accuracy vs naive linear quantization

👉 In practice: this is the recommended default for LLMs.

Putting it all together

This config means:

“Load the model in highly compressed 4-bit form, use an advanced quantization scheme (NF4), apply extra compression (double quant), and do computations in bfloat16 for stability.”

When you’d use this

Typical scenarios:

Running large LLMs on limited VRAM (e.g., 8–24 GB GPUs)
Fine-tuning with methods like LoRA / QLoRA
Inference where slight precision loss is acceptable
Trade-offs

Pros

Massive memory reduction
Enables larger models on smaller hardware

Cons

Slight loss in accuracy
Small compute overhead from dequantization
Not ideal for tasks requiring high numerical precision